In [3]:
%load_ext autoreload
%autoreload 2

import numpy as np
from pathlib import Path
import sys

PROJECT_DIR = str(Path.cwd().parent) + "/"
sys.path.append(PROJECT_DIR)

from mpp_project.end_game_solver import solve_endgame_dp

print("🧪 DÉMARRAGE DU LABORATOIRE DE TESTS UNITAIRES DP")

# ==========================================
# 1. CRÉATION DU MATCH "PIPEAU" (t=31)
# ==========================================
true_proba = np.array([0.60, 0.25, 0.15], dtype=np.float32) 
crowd = np.array([0.70, 0.20, 0.10], dtype=np.float32)
gains = np.array([20, 50, 90], dtype=np.int32) 

match_probs = np.zeros((32, 3), dtype=np.float32)
match_probs[31] = true_proba

crowds = np.zeros((32, 3), dtype=np.float32)
crowds[31] = crowd

gains_1N2 = np.zeros((32, 3), dtype=np.int32)
gains_1N2[31] = gains

p_empirique_1D = np.zeros((32, 3, 250), dtype=np.float32)
p_empirique_1D[31, :, 0] = 1.0 

alphas = np.ones(32, dtype=np.float32)
roles_mock = np.full(32, -1, dtype=np.int32)

# ==========================================
# 2. L'ÉTAT TERMINAL (CORRIGÉ)
# ==========================================
V_term = np.zeros((501, 501, 2, 2, 2, 2), dtype=np.float32)
for g1 in range(501):
    for g2 in range(501):
        # Gap positif = J'ai de l'avance. Donc val >= 0 donne la victoire.
        if (g1 - 250) >= 0 and (g2 - 250) >= 0:
            V_term[g1, g2, :, :, :, :] = 1.0

# ==========================================
# 3. EXÉCUTION DE L'ORACLE
# ==========================================
print("🧠 Lancement de la Programmation Dynamique...")
V_history = solve_endgame_dp(
    match_probs, crowds, gains_1N2, p_empirique_1D, alphas, 
    roles_mock, roles_mock, roles_mock, V_term, stop_t=31
)

Q_table = V_history[31]

# ==========================================
# 4. LES TESTS
# ==========================================
print("\n" + "="*50)
print("📊 RÉSULTATS DES TESTS (Sans Booster)")
print("="*50)

# On met le Peloton à +250 d'avance pour nous (Sécurité totale, on ne joue que contre Bob)
idx_g2_safe = 500  

def test_scenario(nom, gap_reel_avec_bob, index_choix_attendu, calcul_theorique):
    # La grille divise les points par 2 pour ne pas déborder avec les boosters
    gap_grid = int(gap_reel_avec_bob / 2.0)
    idx_g1 = gap_grid + 250
    
    wr_max_obtenu = Q_table[idx_g1, idx_g2_safe, 0, 0, 0, 0]
    
    print(f"▶️ TEST : {nom}")
    print(f"   Mon Avance sur Bob  : {gap_reel_avec_bob} pts")
    print(f"   WR Théorique Attendu: {calcul_theorique*100:.2f}%")
    print(f"   WR Calculé DP       : {wr_max_obtenu*100:.2f}%")
    
    if abs(wr_max_obtenu - calcul_theorique) < 0.001:
        print("   ✅ TEST RÉUSSI")
    else:
        print("   ❌ ÉCHEC DU TEST")
    print("-" * 50)


# --- SCÉNARIO 1 : MISSION IMPOSSIBLE ---
# Bob a 100 pts d'avance (Notre Gap = -100). Le gain max est de 90.
test_scenario("Mission Impossible", gap_reel_avec_bob=-100, index_choix_attendu=None, calcul_theorique=0.0)

# --- SCÉNARIO 2 : LE MIRACLE DE L'OUTSIDER ---
# Bob a 70 pts d'avance (-70). Seul l'outsider (90) peut nous sauver.
wr_theo_out = true_proba[2] * (1.0 - crowd[2])
test_scenario("Seul l'Outsider", gap_reel_avec_bob=-70, index_choix_attendu=2, calcul_theorique=wr_theo_out)

# --- SCÉNARIO 3 : LE NUL OU L'OUTSIDER ---
# Bob a 40 pts d'avance (-40). Nul (50) ou Outsider (90) nous sauvent.
wr_theo_nul = true_proba[1] * (1.0 - crowd[1])
wr_theo_total_40 = max(wr_theo_nul, wr_theo_out) 
test_scenario("Le Nul ou Outsider", gap_reel_avec_bob=-40, index_choix_attendu=1, calcul_theorique=wr_theo_total_40)

# --- SCÉNARIO 4 : ON EST DEVANT ---
# On a 10 pts d'avance (+10). On joue le Favori (Safe). 
# Bob doit prendre plus de risque et avoir raison pour nous passer devant.
wr_theo_safe = 1.0 - (true_proba[1]*crowd[1] + true_proba[2]*crowd[2]) 
test_scenario("Couverture du Leader", gap_reel_avec_bob=10, index_choix_attendu=0, calcul_theorique=wr_theo_safe)

🧪 DÉMARRAGE DU LABORATOIRE DE TESTS UNITAIRES DP
🧠 Lancement de la Programmation Dynamique...

📊 RÉSULTATS DES TESTS (Sans Booster)
▶️ TEST : Mission Impossible
   Mon Avance sur Bob  : -100 pts
   WR Théorique Attendu: 0.00%
   WR Calculé DP       : 0.00%
   ✅ TEST RÉUSSI
--------------------------------------------------
▶️ TEST : Seul l'Outsider
   Mon Avance sur Bob  : -70 pts
   WR Théorique Attendu: 13.50%
   WR Calculé DP       : 13.50%
   ✅ TEST RÉUSSI
--------------------------------------------------
▶️ TEST : Le Nul ou Outsider
   Mon Avance sur Bob  : -40 pts
   WR Théorique Attendu: 20.00%
   WR Calculé DP       : 20.00%
   ✅ TEST RÉUSSI
--------------------------------------------------
▶️ TEST : Couverture du Leader
   Mon Avance sur Bob  : 10 pts
   WR Théorique Attendu: 93.50%
   WR Calculé DP       : 93.50%
   ✅ TEST RÉUSSI
--------------------------------------------------


In [4]:
print("🧪 DÉMARRAGE DU LABORATOIRE DE TESTS UNITAIRES DP (AVEC BOOSTER)")

# ==========================================
# 1. CRÉATION DU MATCH "PIPEAU" (t=31)
# ==========================================
true_proba = np.array([0.60, 0.25, 0.15], dtype=np.float32) 
crowd = np.array([0.70, 0.20, 0.10], dtype=np.float32)
gains = np.array([20, 50, 90], dtype=np.int32) 

match_probs = np.zeros((32, 3), dtype=np.float32)
match_probs[31] = true_proba

crowds = np.zeros((32, 3), dtype=np.float32)
crowds[31] = crowd

gains_1N2 = np.zeros((32, 3), dtype=np.int32)
gains_1N2[31] = gains

p_empirique_1D = np.zeros((32, 3, 250), dtype=np.float32)
p_empirique_1D[31, :, 0] = 1.0 

alphas = np.ones(32, dtype=np.float32)
roles_mock = np.full(32, -1, dtype=np.int32)

# ==========================================
# 2. L'ÉTAT TERMINAL
# ==========================================
V_term = np.zeros((501, 501, 2, 2, 2, 2), dtype=np.float32)
for g1 in range(501):
    for g2 in range(501):
        if (g1 - 250) >= 0 and (g2 - 250) >= 0:
            V_term[g1, g2, :, :, :, :] = 1.0

# ==========================================
# 3. EXÉCUTION DE L'ORACLE
# ==========================================
print("🧠 Lancement de la Programmation Dynamique...")
V_history = solve_endgame_dp(
    match_probs, crowds, gains_1N2, p_empirique_1D, alphas, 
    roles_mock, roles_mock, roles_mock, V_term, stop_t=31
)

Q_table = V_history[31]

# ==========================================
# 4. LES TESTS (DOUBLE DIMENSION : SANS ET AVEC BOOSTER)
# ==========================================
print("\n" + "="*50)
print("📊 RÉSULTATS DES TESTS (Analyse Base vs x2)")
print("="*50)

idx_g2_safe = 500  

def test_scenario_complet(nom, gap_reel_avec_bob, theo_sans_booster, theo_avec_booster):
    gap_grid = int(gap_reel_avec_bob / 2.0)
    idx_g1 = gap_grid + 250
    
    # Lecture des deux dimensions (0 = Sans Booster, 1 = Booster Dispo)
    wr_sans_b = Q_table[idx_g1, idx_g2_safe, 0, 0, 0, 0]
    wr_avec_b = Q_table[idx_g1, idx_g2_safe, 1, 0, 0, 0]
    
    print(f"▶️ TEST : {nom}")
    print(f"   Gap  : {gap_reel_avec_bob} pts (Bob est devant de {-gap_reel_avec_bob} pts)")
    print(f"   [Base] Théorie : {theo_sans_booster*100:05.2f}% | Calcul DP : {wr_sans_b*100:05.2f}%")
    print(f"   [ x2 ] Théorie : {theo_avec_booster*100:05.2f}% | Calcul DP : {wr_avec_b*100:05.2f}%")
    
    reussi = abs(wr_sans_b - theo_sans_booster) < 0.001 and abs(wr_avec_b - theo_avec_booster) < 0.001
    if reussi:
        print("   ✅ TEST RÉUSSI")
    else:
        print("   ❌ ÉCHEC DU TEST")
    print("-" * 50)


# --- SCÉNARIO 1 : MISSION IMPOSSIBLE (MÊME AVEC BOOSTER) ---
# Bob a 200 pts d'avance. Gain Max x2 = 180. Perdu d'avance.
test_scenario_complet(
    "Mission Impossible absolue", 
    gap_reel_avec_bob=-200, 
    theo_sans_booster=0.0, 
    theo_avec_booster=0.0
)

# --- SCÉNARIO 2 : L'OUTSIDER x2 EST OBLIGATOIRE ---
# Bob a 150 pts d'avance. Un outsider normal (90) ne suffit pas. 
# Seul l'outsider x2 (180) passe, à condition que Bob ne le joue pas (sinon on prend 180, Bob prend 90, on reste à -60 = Perdu).
wr_theo_out_seul = true_proba[2] * (1.0 - crowd[2])
test_scenario_complet(
    "Sauvetage In Extremis (Outsider x2 nécessaire)", 
    gap_reel_avec_bob=-150, 
    theo_sans_booster=0.0, 
    theo_avec_booster=wr_theo_out_seul # 13.50%
)

# --- SCÉNARIO 3 : LE CASSE DU SIÈCLE (LE NUL x2 > OUTSIDER x2) ---
# Bob a 95 pts d'avance. Aucun pari simple ne passe (max 90).
# L'outsider x2 donne 180 (passe si Bob joue pas -> WR=13.5%).
# Mais le Nul x2 donne 100 ! S'il passe et que Bob ne l'a pas joué, on gagne de justesse (+5).
# Le WR du Nul (25% * 80% = 20%) est supérieur à celui de l'outsider (13.5%).
# L'Oracle DOIT trouver que le meilleur levier est le Nul x2 !
wr_theo_nul_seul = true_proba[1] * (1.0 - crowd[1])
test_scenario_complet(
    "Le Levier Optimal (Nul x2 meilleur qu'Outsider)", 
    gap_reel_avec_bob=-95, 
    theo_sans_booster=0.0, 
    theo_avec_booster=wr_theo_nul_seul # 20.00%
)

# --- SCÉNARIO 4 : LE BOOSTER ANNULE LE RISQUE ADVERSE ---
# Bob a 40 pts d'avance. 
# Sans booster, on doit jouer le Nul (WR=20%) pour le doubler (si Bob ne le joue pas).
# AVEC booster, si on joue le Nul x2 (100 pts), on gagne MÊME SI Bob joue aussi le Nul (100 - 50 = +50, Gap comblé) !
# Le Nul x2 passe donc de 20% à 25% de WR.
wr_theo_nul_inconditionnel = true_proba[1] * 1.0
test_scenario_complet(
    "Booster de Couverture (Le Nul x2 immunise contre Bob)", 
    gap_reel_avec_bob=-40, 
    theo_sans_booster=wr_theo_nul_seul, # 20.00%
    theo_avec_booster=wr_theo_nul_inconditionnel # 25.00%
)

🧪 DÉMARRAGE DU LABORATOIRE DE TESTS UNITAIRES DP (AVEC BOOSTER)
🧠 Lancement de la Programmation Dynamique...

📊 RÉSULTATS DES TESTS (Analyse Base vs x2)
▶️ TEST : Mission Impossible absolue
   Gap  : -200 pts (Bob est devant de 200 pts)
   [Base] Théorie : 00.00% | Calcul DP : 00.00%
   [ x2 ] Théorie : 00.00% | Calcul DP : 00.00%
   ✅ TEST RÉUSSI
--------------------------------------------------
▶️ TEST : Sauvetage In Extremis (Outsider x2 nécessaire)
   Gap  : -150 pts (Bob est devant de 150 pts)
   [Base] Théorie : 00.00% | Calcul DP : 00.00%
   [ x2 ] Théorie : 13.50% | Calcul DP : 13.50%
   ✅ TEST RÉUSSI
--------------------------------------------------
▶️ TEST : Le Levier Optimal (Nul x2 meilleur qu'Outsider)
   Gap  : -95 pts (Bob est devant de 95 pts)
   [Base] Théorie : 00.00% | Calcul DP : 00.00%
   [ x2 ] Théorie : 20.00% | Calcul DP : 20.00%
   ✅ TEST RÉUSSI
--------------------------------------------------
▶️ TEST : Booster de Couverture (Le Nul x2 immunise contre Bob)
